# 02 — Entraîner les modèles avec PyTorch

On utilise les données du notebook 01 : `data_0.2_TNaugmented`, avec `train_0.8`, `val_0.2` et `test`. Les images augmentées sont déjà dans `train_0.8/NORMAL`.

On entraîne EfficientNetB0 en deux phases, puis un CNN sans préentraînement pour comparaison. Les modèles ont une sortie : un score converti en probabilité de PNEUMONIA. Les cellules d'entraînement lancent réellement les calculs lorsqu'on les exécute ; elles ne sont pas exécutées lors de la création de ce notebook.

Ce portage reprend la démarche Keras, pas des résultats identiques : poids préentraînés, prétraitement et couches dégelées diffèrent. Les deux modèles PyTorch utilisent ici exactement les mêmes splits. Le test ne sert ni à choisir les poids ni à arrêter l'entraînement.

## Installer les bibliothèques

À exécuter si les bibliothèques ne sont pas installées dans le noyau choisi.

In [1]:
%pip install torch torchvision scikit-learn matplotlib Pillow

Note: you may need to restart the kernel to use updated packages.


## Chemins et paramètres

Démarre ce notebook depuis `PROJET_PNEUMONIA_PYTORCH`, comme le 01. Le dossier de résultats est nouveau à chaque exécution de cette cellule, pour conserver les anciens essais. Pour un premier essai court, tu peux réduire les nombres d’époques à 1.

In [52]:
from pathlib import Path
from datetime import datetime

random_state = 42
data_loc = Path("data_nouveau_split2")
dossier_train = data_loc / "train"
dossier_val = data_loc / "val"
dossier_test = data_loc / "test"
epochs_phase1 = 10
epochs_phase2 = 2
epochs_cnn = 30

## Choisir CPU ou GPU

CUDA est utilisé si disponible, sinon le GPU Apple (MPS), sinon le CPU. La graine stabilise les choix aléatoires, sans garantir des résultats identiques entre tous les appareils.

In [39]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using gpu: %s ' % torch.cuda.is_available())

Using gpu: False 


## Préparer les images pour EfficientNet

Les augmentations ont déjà été enregistrées dans le 01 : on ne rajoute pas de recadrage aléatoire ici. On redimensionne les images à 224 × 224 et on applique la normalisation attendue par les poids ImageNet. Ce choix conserve toute l’image, mais peut modifier ses proportions ; le prétraitement officiel des poids utilise plutôt redimensionnement et recadrage central.

In [40]:
from torchvision.transforms import v2

transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## Charger les trois datasets

ImageFolder lit les images en RGB et attribue une classe à chaque sous-dossier. On vérifie que NORMAL vaut 0 et PNEUMONIA vaut 1, sans troisième classe.

In [41]:
from torchvision.datasets import ImageFolder

train_data = ImageFolder(dossier_train, transform=transform)
val_data = ImageFolder(dossier_val, transform=transform)
test_data = ImageFolder(dossier_test, transform=transform)

classes = ["NORMAL", "PNEUMONIA"]

#verif
for dataset in (train_data, val_data, test_data):
    print(dataset.class_to_idx == {"NORMAL": 0, "PNEUMONIA": 1})


True
True
True


##  Création des batchs 

DataLoader fournit des lots d’images. 

In [42]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

## Construire EfficientNetB0

Le premier appel télécharge les poids ImageNet si nécessaire. On remplace la tête par 256 neurones, un dropout et une sortie. La sortie est un logit : on applique sigmoid seulement pour obtenir une probabilité. On gèle le réseau de caractéristiques pour la première phase.

In [43]:
from torch import nn
import torchvision.models as models

weights = models.EfficientNet_B0_Weights.DEFAULT # poids pré entraînés sur imageNet

model = models.efficientnet_b0(weights=weights)

model.classifier = nn.Sequential( #nouvelle tete
    nn.Linear(1280, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 1),
)

model = model.to(device)

for parameter in model.features.parameters():
    parameter.requires_grad = False # On gèle les poids du modèle EfficientNet pré-entraîné

model.features = conserve ses poids pré entrainés 

## Définir la perte

BCEWithLogitsLoss combine sigmoid et perte binaire de façon numériquement stable. Les labels sont convertis en nombres décimaux dans la boucle. Une pénalité L2 sera appliquée au premier Linear de la tête, comme dans le projet Keras.

In [44]:
criterion = nn.BCEWithLogitsLoss()

def classification_loss(model, logits, labels):
    loss = criterion(logits, labels)
    return loss + 1e-4 * model.classifier[0].weight.square().sum()

## evaluation

In [45]:
def test_CNN(modele, batchs_test, critere):  

    modele.eval()  # Active le mode évaluation du modèle (notamment désactivation du dropout).

    #initialisation
    test_loss = 0  
    correct = 0  
    with torch.no_grad():  #désactiver le calcul des gradients 

        for data, target in batchs_test : 
            data = data.to(device)  
            target = target.to(device) 

            output = modele(data).flatten()
            test_loss += critere(output, target.float()).item()*len(target) 

            pred = torch.sigmoid(output) >= 0.5  #prédit PNEUMONIA si la probabilité atteint 0,5
            correct += pred.eq(target).sum()  # Compare les prédictions aux labels et ajoute le nombre de réponses correctes.

        test_loss /= len(batchs_test.dataset) 
        print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(test_loss, correct, len(batchs_test.dataset), 100. * correct / len(batchs_test.dataset)))
        return test_loss

## 10. Répéter les époques et garder le meilleur modèle

In [46]:
from tqdm.auto import tqdm

def train_CNN(model, trainloader, val_loader, loss_fn, n_epochs, optimizer):

    meilleure_loss = test_CNN(model, val_loader, loss_fn)
    torch.save(model.state_dict(), "meilleur_modele.pth")

    for epoch in tqdm(range(n_epochs), desc="Progression totale"):

        model.train()
        model.features.eval()

        running_loss = 0.0
        for batch_idx, (inputs, targets) in enumerate(tqdm(trainloader, desc=f"Époque {epoch + 1}/{n_epochs}", leave=False)):

            optimizer.zero_grad()
            inputs = inputs.to(device)
            targets = targets.to(device) 
            outputs = model(inputs).flatten()

            loss = loss_fn(outputs, targets.float())
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(targets)

        val_loss = test_CNN(model, val_loader, loss_fn)

        if val_loss < meilleure_loss:
            meilleure_loss = val_loss
            torch.save(model.state_dict(), "meilleur_modele.pth")

    # print statistics
    print(f'epoch : {epoch + 1} loss: {running_loss / len(trainloader.dataset):.3f}')

    model.load_state_dict(
    torch.load("meilleur_modele.pth", map_location=device, weights_only=True)
)

## apprendre la tête

Cette cellule lance l’entraînement. Le backbone reste gelé ; seuls les paramètres de la tête apprennent.

In [47]:
optimizer = torch.optim.Adam(model.classifier.parameters(),lr=0.001)
train_CNN(model,train_loader,val_loader,criterion,epochs_phase1,optimizer)


Test set: Avg. loss: 0.6693, Accuracy: 682/874 (78%)



Progression totale:   0%|          | 0/10 [00:00<?, ?it/s]

Époque 1/10:   0%|          | 0/128 [00:00<?, ?it/s]


Test set: Avg. loss: 0.0884, Accuracy: 845/874 (97%)



Époque 2/10:   0%|          | 0/128 [00:00<?, ?it/s]

...


Test set: Avg. loss: 0.1151, Accuracy: 838/874 (96%)



Époque 9/10:   0%|          | 0/128 [00:00<?, ?it/s]


Test set: Avg. loss: 0.0666, Accuracy: 854/874 (98%)



Époque 10/10:   0%|          | 0/128 [00:00<?, ?it/s]


Test set: Avg. loss: 0.0666, Accuracy: 850/874 (97%)

epoch : 10 loss: 0.093


##  ajuster les derniers blocs

On dégèle les deux derniers blocs de `features` et on réduit le taux d’apprentissage.

In [49]:
for parameter in model.features[-2:].parameters():
    parameter.requires_grad = True

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),lr=0.00001)

train_CNN(model, train_loader, val_loader, criterion, epochs_phase2, optimizer)


Test set: Avg. loss: 0.0666, Accuracy: 854/874 (98%)



Progression totale:   0%|          | 0/2 [00:00<?, ?it/s]

Époque 1/2:   0%|          | 0/128 [00:00<?, ?it/s]


Test set: Avg. loss: 0.0579, Accuracy: 854/874 (98%)



Époque 2/2:   0%|          | 0/128 [00:00<?, ?it/s]


Test set: Avg. loss: 0.0602, Accuracy: 855/874 (98%)

epoch : 2 loss: 0.080


## 15. Évaluation finale sur le test

On utilise les meilleurs poids et un seuil fixé à 0.5. La matrice de confusion a les vraies classes en lignes et les prédictions en colonnes. Le notebook 03 pourra ensuite choisir un autre seuil sur validation uniquement.

In [51]:
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

def evaluate(model, loader, critere):
    model.eval()
    labels, probabilities = [], []
    total_loss = 0

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)
            outputs = model(images).flatten()

            total_loss += critere(outputs, targets.float()).item() * len(targets)
            labels.extend(targets.cpu().tolist())
            probabilities.extend(torch.sigmoid(outputs).cpu().tolist())

    # La loss et l’AUC ne dépendent pas du seuil.
    print(f"Loss : {total_loss / len(labels):.4f}")
    print(f"AUC : {roc_auc_score(labels, probabilities):.4f}")

    for seuil in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
        predictions = [int(p >= seuil) for p in probabilities]

        print(f"\nSeuil : {seuil}")
        print(f"Accuracy : {accuracy_score(labels, predictions):.4f}")
        print(classification_report(
            labels, predictions, labels=[0, 1],
            target_names=["NORMAL", "PNEUMONIA"],
            digits=4, zero_division=0
        ))

evaluate(model, val_loader, criterion)

Loss : 0.0579
AUC : 0.9972

Seuil : 0.3
Accuracy : 0.9725
              precision    recall  f1-score   support

      NORMAL     0.9776    0.9198    0.9478       237
   PNEUMONIA     0.9708    0.9922    0.9814       637

    accuracy                         0.9725       874
   macro avg     0.9742    0.9560    0.9646       874
weighted avg     0.9726    0.9725    0.9723       874


Seuil : 0.4
Accuracy : 0.9760
              precision    recall  f1-score   support

      NORMAL     0.9779    0.9325    0.9546       237
   PNEUMONIA     0.9753    0.9922    0.9837       637

    accuracy                         0.9760       874
   macro avg     0.9766    0.9623    0.9692       874
weighted avg     0.9760    0.9760    0.9758       874


Seuil : 0.5
Accuracy : 0.9771
              precision    recall  f1-score   support

      NORMAL     0.9697    0.9451    0.9573       237
   PNEUMONIA     0.9798    0.9890    0.9844       637

    accuracy                         0.9771       874
   macro

## 16. CNN de comparaison : images à 128 × 128

Cette seconde expérience reprend les quatre blocs du CNN Keras. Les mêmes fichiers sont utilisés, mais les pixels restent entre 0 et 1, sans normalisation ImageNet.

In [ ]:
cnn_preparation = v2.Compose([
    v2.ToImage(), v2.Resize((128, 128), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
])
cnn_train = ImageFolder(dossier_train, transform=cnn_preparation)
cnn_val = ImageFolder(dossier_val, transform=cnn_preparation)
cnn_test = ImageFolder(dossier_test, transform=cnn_preparation)
cnn_train_loader = DataLoader(cnn_train, batch_size=batch_size, shuffle=True)
cnn_val_loader = DataLoader(cnn_val, batch_size=batch_size, shuffle=False)
cnn_test_loader = DataLoader(cnn_test, batch_size=batch_size, shuffle=False)

## 17. Construire le CNN

Convolution, ReLU, BatchNorm et réduction spatiale composent chaque bloc. Le dernier bloc est suivi d’une moyenne spatiale puis de la même tête 256 → 1. Aucun poids préentraîné n’est chargé.

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        blocks = []
        channels = 3
        for filters in (32, 64, 128, 256):
            blocks.extend([nn.Conv2d(channels, filters, 3, padding=1),
                           nn.ReLU(), nn.BatchNorm2d(filters)])
            if filters != 256:
                blocks.append(nn.MaxPool2d(2))
            channels = filters
        self.convolutions = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 1))

    def forward(self, images):
        images = self.convolutions(images)
        images = self.pool(images).flatten(1)
        return self.classifier(images)

torch.manual_seed(random_state)
custom_model = CustomCNN().to(device)

## 18. Entraîner et évaluer le CNN

Cette cellule lance la seconde expérience. La fonction fit recharge le meilleur checkpoint avant le test. Les métriques comparent ici deux architectures sur le même découpage ; elles ne doivent pas servir à ajuster les modèles sur le test.

In [ ]:
cnn_history, cnn_best_auc = fit(custom_model, cnn_train_loader, cnn_val_loader,
    epochs_cnn, 1e-3, resultats / "best_custom_cnn.pth", "CNN")
(resultats / "history_cnn.json").write_text(json.dumps(cnn_history, indent=2))
plot_history(cnn_history, resultats / "learning_cnn.png")
cnn_metrics = evaluate(custom_model, cnn_test_loader, "cnn")
print("EfficientNet :", efficientnet_metrics)
print("CNN :", cnn_metrics)